In [6]:
import pandas as pd
import numpy as np
import ast
import re

In [7]:
# since df_results is too big, put locally in your current folder (since its not on github)
csv_path = "df_results.csv"
df_raw = pd.read_csv(csv_path)
print(df_raw.shape)
print(df_raw.columns.tolist())

# if you wanna see the shape, but loading all is too long
# df = pd.read_csv("df_results.csv", nrows=1)
# print(df.head())
# print(df.info())

/var/folders/8z/zcsmsmvn04b4vm1nxr6_4v480000gq/T/ipykernel_95681/1943761279.py:3: DtypeWarning: Columns (0: postal_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(csv_path)


(227932, 224)
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'postal_code', 'latitude', 'longitude', 'mrt_nearest', 'mrt_nearby', 'bus_nearest', 'bus_nearby', 'store_nearest', 'store_nearby', 'food_nearest', 'food_nearby', 'health_nearest', 'health_nearby', 'restaurant_nearest', 'restaurant_nearby', 'hospital_nearest', 'hospital_nearby', 'lodging_nearest', 'lodging_nearby', 'finance_nearest', 'finance_nearby', 'cafe_nearest', 'cafe_nearby', 'convenience_store_nearest', 'convenience_store_nearby', 'clothing_store_nearest', 'clothing_store_nearby', 'atm_nearest', 'atm_nearby', 'shopping_mall_nearest', 'shopping_mall_nearby', 'grocery_or_supermarket_nearest', 'grocery_or_supermarket_nearby', 'home_goods_store_nearest', 'home_goods_store_nearby', 'school_nearest', 'school_nearby', 'bakery_nearest', 'bakery_nearby', 'beauty_salon_nearest', 'beauty_salon_nearby', 'transit_station

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

df_raw.head()

## Data Processing

In [37]:
# trimmed down the data, its too many columns
# TODO: add distance / time to parent's home & office
base_cols = [
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "flat_model",
    "remaining_lease",
    "resale_price",
    "latitude", # replaces postal code
    "longitude",
]

# TODO: we can use poi_nearby (density might be a better indicator)
poi_nearest_cols = [
    "mrt_nearest",
    "bus_nearest",
    "shopping_mall_nearest",
    "grocery_or_supermarket_nearest",
    #"supermarket_nearest", 
    "park_nearest",
    "doctor_nearest",
    "pharmacy_nearest",
    "cafe_nearest",
    "restaurant_nearest",
    "gym_nearest"
]

def extract_distance(cell):
    if pd.isna(cell):
        return np.nan
    
    if isinstance(cell, (int, float)):
        return float(cell)
    
    try:
        obj = ast.literal_eval(cell)
        if isinstance(obj, dict):
            return obj.get("distance_m", np.nan)
    except Exception:
        pass
    
    return np.nan
    

In [40]:
poi_nearby_cols = [
    "cafe_nearby",
    "restaurant_nearby",
    "bus_nearby",
]

def count_within_radius(cell, radius_m=500):
    if pd.isna(cell):
        return 0
    try:
        entries = ast.literal_eval(cell) if isinstance(cell, str) else cell
        return sum(1 for e in entries if e.get("distance_m", float("inf")) < radius_m)
    except Exception:
        return 0

In [41]:
df_flats = df_raw[base_cols + poi_nearby_cols + poi_nearest_cols].copy()

for col in poi_nearest_cols:
    new_col = col.replace("_nearest", "_distance_m")
    df_flats[new_col] = df_flats[col].apply(extract_distance)

for col in poi_nearby_cols:
    new_col = col.replace("_nearby", f"_count_{500}m")
    df_flats[new_col] = df_flats[col].apply(count_within_radius)

df_flats = df_flats.drop(columns=poi_nearest_cols+poi_nearby_cols)

print(df_flats.shape)
df_flats.head()

(227932, 23)


,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,remaining_lease,resale_price,latitude,longitude,mrt_distance_m,bus_distance_m,shopping_mall_distance_m,grocery_or_supermarket_distance_m,park_distance_m,doctor_distance_m,pharmacy_distance_m,cafe_distance_m,restaurant_distance_m,gym_distance_m,cafe_count_500m,restaurant_count_500m,bus_count_500m
0,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,61 years 04 months,232000.0,1.362005,103.853880,926.897893,91.924399,1032.440827,298.943356,702.372551,759.478958,719.191844,942.041113,675.805030,685.487900,0,0,14
1,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,60 years 07 months,250000.0,1.370966,103.838202,1316.429566,166.312373,873.663121,374.856579,445.644348,762.518468,1004.302877,1054.606053,259.598745,509.849980,0,2,11
2,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,62 years 05 months,262000.0,1.380709,103.835368,1071.182879,129.344650,1531.464976,874.666963,738.579066,1704.455313,1734.705553,1711.519153,853.948798,1629.408340,0,0,15
3,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,62 years 01 month,265000.0,1.366201,103.857201,880.423864,69.452838,875.833046,574.970520,922.732931,881.505494,262.133206,893.065336,176.628039,764.432883,0,1,10
4,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,62 years 05 months,265000.0,1.381041,103.835132,1094.012991,149.897908,1575.311125,919.965695,783.742843,1749.806066,1778.749292,1754.576301,897.963781,1673.219891,0,0,11


In [35]:
df_flats[df_flats["cafe_count_500m"]>0].shape

(111081, 24)

In [42]:
def parse_remaining_lease_to_years(x):
    if pd.isna(x):
        return np.nan
    
    text = str(x).lower()
    years = 0
    months = 0
    
    y_match = re.search(r"(\d+)\s*year", text)
    m_match = re.search(r"(\d+)\s*month", text)
    
    if y_match:
        years = int(y_match.group(1))
    if m_match:
        months = int(m_match.group(1))
    
    return years + months / 12

df_flats["remaining_lease_years"] = df_flats["remaining_lease"].apply(parse_remaining_lease_to_years)

In [43]:
def parse_storey_range_mid(x):
    if pd.isna(x):
        return np.nan
    try:
        parts = str(x).split(" TO ")
        low = int(parts[0])
        high = int(parts[1])
        return (low + high) / 2
    except Exception:
        return np.nan

df_flats["storey_mid"] = df_flats["storey_range"].apply(parse_storey_range_mid)

In [44]:
df_flats["flat_id"] = (
    df_flats["block"].astype(str) + "_" +
    df_flats["street_name"].astype(str) + "_" +
    df_flats["flat_type"].astype(str) + "_"
)

In [45]:
flat_feature_cols = [
    "flat_id",
    "flat_type",
    "flat_model",
    "floor_area_sqm",
    "resale_price",
    "latitude",
    "longitude",
    "remaining_lease_years",
    "storey_mid",

    # transport
    "mrt_distance_m",
    "bus_distance_m",
    "bus_count_500m",

    # convenience
    "shopping_mall_distance_m",
    "grocery_or_supermarket_distance_m",

    # lifestyle
    "park_distance_m",
    "cafe_distance_m",
    "restaurant_distance_m",
    "gym_distance_m",
    "cafe_count_500m",
    "restaurant_count_500m",

    # health
    "doctor_distance_m",
    "pharmacy_distance_m",
]

df_flats_clean = df_flats[flat_feature_cols].copy()
df_flats_clean.head()

,flat_id,flat_type,flat_model,floor_area_sqm,resale_price,latitude,longitude,remaining_lease_years,storey_mid,mrt_distance_m,bus_distance_m,bus_count_500m,shopping_mall_distance_m,grocery_or_supermarket_distance_m,park_distance_m,cafe_distance_m,restaurant_distance_m,gym_distance_m,cafe_count_500m,restaurant_count_500m,doctor_distance_m,pharmacy_distance_m
0,406_ANG MO KIO AVE 10_2 ROOM_,2 ROOM,Improved,44.0,232000.0,1.362005,103.853880,61.333333,11.0,926.897893,91.924399,14,1032.440827,298.943356,702.372551,942.041113,675.805030,685.487900,0,0,759.478958,719.191844
1,108_ANG MO KIO AVE 4_3 ROOM_,3 ROOM,New Generation,67.0,250000.0,1.370966,103.838202,60.583333,2.0,1316.429566,166.312373,11,873.663121,374.856579,445.644348,1054.606053,259.598745,509.849980,0,2,762.518468,1004.302877
2,602_ANG MO KIO AVE 5_3 ROOM_,3 ROOM,New Generation,67.0,262000.0,1.380709,103.835368,62.416667,2.0,1071.182879,129.344650,15,1531.464976,874.666963,738.579066,1711.519153,853.948798,1629.408340,0,0,1704.455313,1734.705553
3,465_ANG MO KIO AVE 10_3 ROOM_,3 ROOM,New Generation,68.0,265000.0,1.366201,103.857201,62.083333,5.0,880.423864,69.452838,10,875.833046,574.970520,922.732931,893.065336,176.628039,764.432883,0,1,881.505494,262.133206
4,601_ANG MO KIO AVE 5_3 ROOM_,3 ROOM,New Generation,67.0,265000.0,1.381041,103.835132,62.416667,2.0,1094.012991,149.897908,11,1575.311125,919.965695,783.742843,1754.576301,897.963781,1673.219891,0,0,1749.806066,1778.749292


## Represent User Preferences : 

In [62]:
user_feature_cols = [
    "user_id",
    "budget_max",
    "preferred_flat_type",
    "min_floor_area",
    "preferred_region",
    "max_mrt_distance_m",
    "max_bus_distance_m",
    "weight_transport",
    "weight_amenities",
    "weight_recreation",
    "weight_health",
    "weight_space",
    "weight_storey",
    "weight_lease",
    "weight_office",
    "weight_parents",
    "parent_lat",
    "parent_lon",
    "office_lat",
    "office_lon",
]

# df_users = pd.DataFrame(users)
# df_users

## Generate dummy user data

In [ ]:
import random

np.random.seed(42)
random.seed(42)

OFFICE_LOCATIONS = [
    # CBD / Raffles Place
    {"name": "Raffles Place", "lat": 1.2841, "lon": 103.8515},
    {"name": "Marina Bay Financial Centre", "lat": 1.2768, "lon": 103.8514},
    {"name": "Tanjong Pagar", "lat": 1.2765, "lon": 103.8448},
    {"name": "Shenton Way", "lat": 1.2784, "lon": 103.8479},
    {"name": "Robinson Road", "lat": 1.2796, "lon": 103.8493},
    # Orchard / River Valley
    {"name": "Orchard", "lat": 1.3048, "lon": 103.8318},
    {"name": "River Valley", "lat": 1.2908, "lon": 103.8365},
    # Bugis / Beach Road
    {"name": "Bugis", "lat": 1.3009, "lon": 103.8560},
    {"name": "Beach Road", "lat": 1.2966, "lon": 103.8600},
    # Paya Lebar / Ubi / Tai Seng
    {"name": "Paya Lebar Quarter", "lat": 1.3175, "lon": 103.8924},
    {"name": "Ubi Techpark", "lat": 1.3298, "lon": 103.8988},
    {"name": "Tai Seng", "lat": 1.3354, "lon": 103.8880},
    {"name": "Macpherson", "lat": 1.3267, "lon": 103.8898},
    # Jurong / West
    {"name": "Jurong Gateway", "lat": 1.3329, "lon": 103.7436},
    {"name": "International Business Park", "lat": 1.3339, "lon": 103.7424},
    {"name": "Jurong East", "lat": 1.3331, "lon": 103.7424},
    {"name": "Boon Lay", "lat": 1.3386, "lon": 103.7060},
    # one-north / Buona Vista
    {"name": "one-north", "lat": 1.2995, "lon": 103.7876},
    {"name": "Buona Vista", "lat": 1.3072, "lon": 103.7902},
    # Woodlands / North
    {"name": "Woodlands Regional Centre", "lat": 1.4370, "lon": 103.7863},
    {"name": "Woodlands Causeway Point area", "lat": 1.4369, "lon": 103.7863},
    # Tampines / East
    {"name": "Tampines Regional Centre", "lat": 1.3406, "lon": 103.9452},
    {"name": "Changi Business Park", "lat": 1.3352, "lon": 103.9637},
    {"name": "Expo", "lat": 1.3352, "lon": 103.9613},
    # Novena / Balestier
    {"name": "Novena", "lat": 1.3200, "lon": 103.8438},
    {"name": "Balestier", "lat": 1.3228, "lon": 103.8476},
    # Kallang / Mountbatten
    {"name": "Kallang Basin", "lat": 1.3101, "lon": 103.8726},
    {"name": "Mountbatten", "lat": 1.3063, "lon": 103.8793},
    # Science / Education clusters
    {"name": "Kent Ridge / NUS", "lat": 1.2966, "lon": 103.7764},
    {"name": "Queenstown / Alexandra", "lat": 1.2780, "lon": 103.8006}
]

real_locations = df_raw[["latitude", "longitude"]].dropna().drop_duplicates().values.tolist()

flat_types = ["2 ROOM", "3 ROOM", "4 ROOM"]
flat_type_probs = [0.45, 0.4, 0.15]

users = []
for i in range(300): # change value if we need more users
    office = random.choice(OFFICE_LOCATIONS)
    parent_home = random.choice(real_locations)

    users.append({
        "user_id": f"U{i+1}",
        "budget_max":          np.random.choice([300000, 350000, 400000, 450000, 500000, 600000]),
        "preferred_flat_type": np.random.choice(flat_types, p=flat_type_probs),
        "weight_transport":    np.random.choice([3, 4, 5], p=[0.3, 0.4, 0.3]),
        "weight_essentials":   np.random.choice([2, 3, 4, 5], p=[0.1, 0.3, 0.4, 0.2]),
        "weight_lifestyle":    np.random.choice([1, 2, 3, 4, 5]),
        "weight_recreation":   np.random.choice([1, 2, 3, 4, 5]),
        "weight_health":       np.random.choice([1, 2, 3, 4, 5], p=[0.3, 0.3, 0.2, 0.1, 0.1]),
        "weight_storey":       np.random.choice([1, 2, 3, 4, 5], p=[0.3, 0.3, 0.2, 0.1, 0.1]),
        "weight_lease":        np.random.choice([1, 2, 3, 4, 5], p=[0.1, 0.2, 0.3, 0.2, 0.2]),
        "weight_office":       np.random.choice([1, 2, 3, 4, 5]),
        "weight_parents":      np.random.choice([1, 2, 3, 4, 5]),
        "parent_lat": parent_home[0],
        "parent_lon": parent_home[1],
        "office_lat": office["lat"],
        "office_lon": office["lon"],
    })

df_users = pd.DataFrame(users)

In [78]:
df_users.head()

,user_id,budget_max,preferred_flat_type,weight_transport,weight_essentials,weight_lifestyle,weight_recreation,weight_health,weight_storey,weight_lease,weight_office,weight_parents,parent_lat,parent_lon,office_lat,office_lon
0,U1,450000,4 ROOM,5,4,2,3,1,2,3,3,5,1.370364,103.890000,1.2768,103.8514
1,U2,350000,3 ROOM,5,2,4,5,3,3,1,1,1,1.277185,103.819516,1.3175,103.8924
2,U3,400000,3 ROOM,3,3,3,4,2,2,1,3,5,1.366339,103.887775,1.3048,103.8318
3,U4,300000,2 ROOM,5,5,2,2,2,1,4,4,4,1.431856,103.775294,1.2768,103.8514
4,U5,500000,3 ROOM,3,3,2,4,2,2,3,5,2,1.356495,103.742713,1.2768,103.8514


In [ ]:
# Comment out cos I think the region doesnt matter that much (we can add in a filter later if necessary anw)
# # because we dont expect users to input their preference lat, long
# # we just ask them to input preferred region and translate here
# region_centers = {
#     "North": (1.4360, 103.7860),
#     "North-East": (1.3880, 103.8920),
#     "Central": (1.3280, 103.8420),
#     "East": (1.3570, 103.9400),
#     "West": (1.3450, 103.7200),
# }

# df_users["preferred_lat"] = df_users["preferred_region"].map(lambda r: region_centers[r][0])
# df_users["preferred_lon"] = df_users["preferred_region"].map(lambda r: region_centers[r][1])

### Generate flats for user to do labelling on

In [72]:
# hard filters for each user
def get_user_candidates(user_row, df_flats):
    """Filter flats that satisfy a user's hard constraints."""
    candidates = df_flats.copy()

    candidates = candidates[candidates["resale_price"] <= user_row["budget_max"]]
    candidates = candidates[candidates["flat_type"].str.upper() == user_row["preferred_flat_type"].upper()]

    return candidates


# Build per-user candidate lists
user_candidates = {}  # {user_id: df of valid flats}

for _, user_row in df_users.iterrows():
    candidates = get_user_candidates(user_row, df_flats_clean)
    user_candidates[user_row["user_id"]] = candidates


### Label Pairwise Choices

In [73]:
from math import radians, sin, cos, sqrt, atan2

def safe_distance_score(dist_m, cap=2000):
    if pd.isna(dist_m):
        return 0.0
    return max(0.0, 1 - (min(dist_m, cap) / cap))

def safe_count_score(count, cap=5):
    if pd.isna(count):
        return 0.0
    return min(count, cap) / cap

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in metres
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))


def compute_scores(candidates, user_row):
    c = candidates.copy()

    # transport: MRT (50%) + bus distance (30%) + bus density (20%)
    c["transport_score"] = (
        c["mrt_distance_m"].apply(safe_distance_score) * 0.5 +
        c["bus_distance_m"].apply(safe_distance_score) * 0.3 +
        c["bus_count_500m"].apply(safe_count_score) * 0.2
    )

    # essentials: supermarket only
    c["essentials_score"] = (
        c["grocery_or_supermarket_distance_m"].apply(safe_distance_score)
    )

    # lifestyle: cafes + restaurants (nearest + density) + mall
    c["lifestyle_score"] = (
        c["restaurant_distance_m"].apply(safe_distance_score) * 0.2 +
        c["restaurant_count_500m"].apply(safe_count_score) * 0.25 +
        c["cafe_distance_m"].apply(safe_distance_score) * 0.15 +
        c["cafe_count_500m"].apply(safe_count_score) * 0.2 +
        c["shopping_mall_distance_m"].apply(safe_distance_score) * 0.2
    )

    # recreation: park + gym
    c["recreation_score"] = (
        c["park_distance_m"].apply(safe_distance_score) * 0.5 +
        c["gym_distance_m"].apply(safe_distance_score) * 0.5
    )

    # health: doctor + pharmacy
    c["health_score"] = (
        c["doctor_distance_m"].apply(safe_distance_score) * 0.6 +
        c["pharmacy_distance_m"].apply(safe_distance_score) * 0.4
    )

    # storey: higher is better, normalised against SG HDB max (50)
    c["storey_score"] = (c["storey_mid"] / 50.0).clip(0, 1)

    # lease: normalised against max new lease (99 years)
    c["lease_score"] = (c["remaining_lease_years"] / 99.0).clip(0, 1)

    # office: distance from flat to user's office (wider cap for commutes)
    c["office_distance_m"] = c.apply(
        lambda r: haversine_distance(
            r["latitude"], r["longitude"],
            user_row["office_lat"], user_row["office_lon"]
        ), axis=1
    )
    c["office_score"] = c["office_distance_m"].apply(
        lambda d: safe_distance_score(d, cap=10000)
    )

    # parents: distance from flat to parents' home
    c["parents_distance_m"] = c.apply(
        lambda r: haversine_distance(
            r["latitude"], r["longitude"],
            user_row["parent_lat"], user_row["parent_lon"]
        ), axis=1
    )
    c["parents_score"] = c["parents_distance_m"].apply(
        lambda d: safe_distance_score(d, cap=10000)
    )

    return c

In [74]:
def compute_raw_relevance(user_row, candidates):
    return (
        user_row["weight_transport"]  * candidates["transport_score"] +
        user_row["weight_essentials"] * candidates["essentials_score"] +
        user_row["weight_lifestyle"]  * candidates["lifestyle_score"] +
        user_row["weight_recreation"] * candidates["recreation_score"] +
        user_row["weight_health"]     * candidates["health_score"] +
        user_row["weight_storey"]     * candidates["storey_score"] +
        user_row["weight_lease"]      * candidates["lease_score"] +
        user_row["weight_office"]     * candidates["office_score"] +
        user_row["weight_parents"]    * candidates["parents_score"]
    )

In [ ]:
def sample_pairs_stratified(group, n_pairs, rng):
    """70% close-score pairs (informative), 30% random pairs (coverage)."""
    group = group.sort_values("raw_relevance").reset_index(drop=True)
    n = len(group)

    idx_a, idx_b = [], []

    # 70% close pairs — nearby in ranked score order
    n_close = int(n_pairs * 0.7)
    for _ in range(n_close):
        i = int(rng.integers(0, n - 1))
        gap = int(rng.integers(1, min(10, n - i)))
        idx_a.append(i)
        idx_b.append(i + gap)

    # 30% random pairs — for coverage
    n_random = n_pairs - n_close
    ra = rng.integers(0, n, size=n_random * 2)
    rb = rng.integers(0, n, size=n_random * 2)
    mask = ra != rb
    idx_a.extend(ra[mask][:n_random].tolist())
    idx_b.extend(rb[mask][:n_random].tolist())

    return idx_a[:n_pairs], idx_b[:n_pairs]


def bradley_terry_label(score_a, score_b, rng, temperature):
    """Probabilistic preference: larger score gap = more confident label."""
    diff = (score_a - score_b) / temperature
    p_a_wins = 1 / (1 + np.exp(-diff))
    return int(rng.random() < p_a_wins)


def generate_pairwise_labels(user_candidates, df_users, score_cols,
                              pairs_per_user=200, random_state=42):
    """
    For each user:
      1. Compute scores and raw_relevance on their candidate flats
      2. Sample pairs (stratified: 70% close-score, 30% random)
      3. Label via Bradley-Terry (probabilistic, human-like) -> Label = 1 if A is preferred over B, 0 otherwise
    """
    rng = np.random.default_rng(random_state)
    user_lookup = df_users.set_index("user_id")

    # calibrate temperature to score spread across all users
    all_scores = []
    for user_id, candidates in user_candidates.items():
        user_row = user_lookup.loc[user_id]
        scored = compute_scores(candidates, user_row)
        all_scores.extend(compute_raw_relevance(user_row, scored).tolist())
    temperature = np.std(all_scores) * 0.25
    print(f"Bradley-Terry temperature: {temperature:.4f}")

    weight_cols = [
        "weight_transport", "weight_essentials", "weight_lifestyle",
        "weight_recreation", "weight_health", "weight_storey",
        "weight_lease", "weight_office", "weight_parents",
    ]

    records = []
    for user_id, candidates in user_candidates.items():
        if len(candidates) < 2:
            continue

        user_row = user_lookup.loc[user_id]
        scored = compute_scores(candidates, user_row).reset_index(drop=True)
        scored["raw_relevance"] = compute_raw_relevance(user_row, scored)

        n_pairs = min(pairs_per_user, len(scored) * (len(scored) - 1) // 2)
        if n_pairs < 1:
            continue

        idx_a, idx_b = sample_pairs_stratified(scored, n_pairs, rng)

        for ia, ib in zip(idx_a, idx_b):
            row_a = scored.iloc[ia]
            row_b = scored.iloc[ib]

            record = {
                "user_id": user_id,
                "flat_id_a": row_a["flat_id"],
                "flat_id_b": row_b["flat_id"],
                "label": bradley_terry_label(
                    row_a["raw_relevance"], row_b["raw_relevance"], rng, temperature
                ),
            }
            for col in weight_cols:
                record[col] = user_row[col]
            for col in score_cols:
                record[f"diff_{col}"] = row_a[col] - row_b[col]

            records.append(record)

    df_pairwise = pd.DataFrame(records)
    print(f"Total pairs: {len(df_pairwise)}")
    print(f"Label balance:\n{df_pairwise['label'].value_counts()}")
    return df_pairwise


score_cols = [
    "transport_score", "essentials_score", "lifestyle_score",
    "recreation_score", "health_score", "storey_score",
    "lease_score", "office_score", "parents_score",
    "mrt_distance_m", "bus_distance_m", "bus_count_500m",
    "grocery_or_supermarket_distance_m", "shopping_mall_distance_m",
    "restaurant_distance_m", "restaurant_count_500m",
    "cafe_distance_m", "cafe_count_500m",
    "park_distance_m", "gym_distance_m",
    "doctor_distance_m", "pharmacy_distance_m",
    "storey_mid", "remaining_lease_years", "floor_area_sqm",
    "office_distance_m", "parents_distance_m",
]

# change pairs per user if we need more labels
df_pairwise = generate_pairwise_labels(
    user_candidates, df_users, score_cols, pairs_per_user=200
)
df_pairwise.head()

Bradley-Terry temperature: 0.6360
Total pairs: 60000
Label balance:
label
0    30073
1    29927
Name: count, dtype: int64


,user_id,flat_id_a,flat_id_b,label,weight_transport,weight_essentials,weight_lifestyle,weight_recreation,weight_health,weight_storey,weight_lease,weight_office,weight_parents,diff_transport_score,diff_essentials_score,diff_lifestyle_score,diff_recreation_score,diff_health_score,diff_storey_score,diff_lease_score,diff_office_score,diff_parents_score,diff_mrt_distance_m,diff_bus_distance_m,diff_bus_count_500m,diff_grocery_or_supermarket_distance_m,diff_shopping_mall_distance_m,diff_restaurant_distance_m,diff_restaurant_count_500m,diff_cafe_distance_m,diff_cafe_count_500m,diff_park_distance_m,diff_gym_distance_m,diff_doctor_distance_m,diff_pharmacy_distance_m,diff_storey_mid,diff_remaining_lease_years,diff_floor_area_sqm,diff_office_distance_m,diff_parents_distance_m
0,U1,669_HOUGANG AVE 8_4 ROOM_,103_HOUGANG AVE 1_4 ROOM_,1,5,4,2,3,1,2,3,3,5,-0.034441,-0.011070,0.134860,0.166108,-0.168916,0.06,0.038721,-0.025755,0.094126,147.242331,-15.796167,6,22.139827,310.885499,71.195296,1,-574.244517,2,472.719151,-1137.149655,318.883704,366.254051,3.0,3.833333,-5.0,1594.246497,-941.258949
1,U1,424C_YISHUN AVE 11_4 ROOM_,429B_YISHUN AVE 11_4 ROOM_,0,5,4,2,3,1,2,3,3,5,0.003318,0.072487,0.016139,0.033964,0.053506,0.00,0.000000,0.000000,-0.014118,-84.286216,118.356126,4,-144.974363,70.505509,-169.367907,0,-83.376508,0,-60.364779,-123.985874,-112.612030,-98.610306,0.0,0.000000,0.0,171.680037,141.175071
2,U1,418_CANBERRA RD_4 ROOM_,508A_WELLINGTON CIRCLE_4 ROOM_,1,5,4,2,3,1,2,3,3,5,0.001664,0.014744,0.095689,-0.027982,0.010014,0.00,-0.034512,0.000000,0.000000,-11.205325,7.582653,1,-29.487386,-9.068924,-16.014805,2,90.930692,1,73.292998,74.118660,-24.244210,-13.702818,0.0,-3.416667,0.0,7.170500,77.027042
3,U1,136_BT BATOK WEST AVE 6_4 ROOM_,291A_BT BATOK ST 24_4 ROOM_,1,5,4,2,3,1,2,3,3,5,0.081584,-0.002807,0.221512,-0.004573,-0.002084,-0.18,-0.132997,0.000000,0.000000,-372.697119,77.268423,-2,5.613314,-376.684924,-277.477277,3,-81.283595,0,139.896394,-121.602443,47.468297,-60.783071,-9.0,-13.166667,-7.0,1480.694784,1095.842484
4,U1,621_CHOA CHU KANG ST 62_4 ROOM_,476D_CHOA CHU KANG AVE 5_4 ROOM_,0,5,4,2,3,1,2,3,3,5,0.157743,-0.064112,0.375389,0.245720,-0.202860,-0.06,-0.173401,0.000000,0.000000,-654.054871,38.469240,-11,128.223667,15.096649,-552.899547,6,111.877528,2,-982.880807,1163.136025,610.726885,98.208540,-3.0,-17.166667,15.0,1057.239362,-674.977657


In [ ]:
pd.set_option('display.max_rows', None)
df_pairwise.head(200)

,user_id,flat_id_a,flat_id_b,label,weight_transport,weight_essentials,weight_lifestyle,weight_recreation,weight_health,weight_storey,weight_lease,weight_office,weight_parents,diff_transport_score,diff_essentials_score,diff_lifestyle_score,diff_recreation_score,diff_health_score,diff_storey_score,diff_lease_score,diff_office_score,diff_parents_score,diff_mrt_distance_m,diff_bus_distance_m,diff_bus_count_500m,diff_grocery_or_supermarket_distance_m,diff_shopping_mall_distance_m,diff_restaurant_distance_m,diff_restaurant_count_500m,diff_cafe_distance_m,diff_cafe_count_500m,diff_park_distance_m,diff_gym_distance_m,diff_doctor_distance_m,diff_pharmacy_distance_m,diff_storey_mid,diff_remaining_lease_years,diff_floor_area_sqm,diff_office_distance_m,diff_parents_distance_m
0,U1,669_HOUGANG AVE 8_4 ROOM_,103_HOUGANG AVE 1_4 ROOM_,1,5,4,2,3,1,2,3,3,5,-0.034441,-0.011070,0.134860,0.166108,-0.168916,0.06,0.038721,-0.025755,0.094126,147.242331,-15.796167,6,22.139827,310.885499,71.195296,1,-574.244517,2,472.719151,-1137.149655,318.883704,366.254051,3.0,3.833333,-5.0,1594.246497,-941.258949
1,U1,424C_YISHUN AVE 11_4 ROOM_,429B_YISHUN AVE 11_4 ROOM_,0,5,4,2,3,1,2,3,3,5,0.003318,0.072487,0.016139,0.033964,0.053506,0.00,0.000000,0.000000,-0.014118,-84.286216,118.356126,4,-144.974363,70.505509,-169.367907,0,-83.376508,0,-60.364779,-123.985874,-112.612030,-98.610306,0.0,0.000000,0.0,171.680037,141.175071
2,U1,418_CANBERRA RD_4 ROOM_,508A_WELLINGTON CIRCLE_4 ROOM_,1,5,4,2,3,1,2,3,3,5,0.001664,0.014744,0.095689,-0.027982,0.010014,0.00,-0.034512,0.000000,0.000000,-11.205325,7.582653,1,-29.487386,-9.068924,-16.014805,2,90.930692,1,73.292998,74.118660,-24.244210,-13.702818,0.0,-3.416667,0.0,7.170500,77.027042
3,U1,136_BT BATOK WEST AVE 6_4 ROOM_,291A_BT BATOK ST 24_4 ROOM_,1,5,4,2,3,1,2,3,3,5,0.081584,-0.002807,0.221512,-0.004573,-0.002084,-0.18,-0.132997,0.000000,0.000000,-372.697119,77.268423,-2,5.613314,-376.684924,-277.477277,3,-81.283595,0,139.896394,-121.602443,47.468297,-60.783071,-9.0,-13.166667,-7.0,1480.694784,1095.842484
4,U1,621_CHOA CHU KANG ST 62_4 ROOM_,476D_CHOA CHU KANG AVE 5_4 ROOM_,0,5,4,2,3,1,2,3,3,5,0.157743,-0.064112,0.375389,0.245720,-0.202860,-0.06,-0.173401,0.000000,0.000000,-654.054871,38.469240,-11,128.223667,15.096649,-552.899547,6,111.877528,2,-982.880807,1163.136025,610.726885,98.208540,-3.0,-17.166667,15.0,1057.239362,-674.977657
5,U1,274_YISHUN ST 22_4 ROOM_,308_YISHUN RING RD_4 ROOM_,0,5,4,2,3,1,2,3,3,5,-0.095867,-0.184073,0.028033,-0.093497,-0.104674,-0.06,-0.023569,0.000000,-0.064917,349.115966,57.255827,-1,368.146320,315.686190,-394.688406,1,398.229863,0,295.297353,78.691814,483.901706,-202.480499,-3.0,-2.333333,-4.0,703.146153,649.165496
6,U1,987A_BUANGKOK GREEN_4 ROOM_,606_HOUGANG AVE 4_4 ROOM_,0,5,4,2,3,1,2,3,3,5,-0.154429,0.089698,-0.066642,-0.100139,-0.319111,0.00,0.297138,0.000000,-0.121739,620.345882,-4.383811,-2,-179.396692,-411.673493,81.798585,0,261.731216,-2,-839.814506,1240.369390,892.522835,256.772205,0.0,29.416667,9.0,726.357784,1217.393544
7,U1,476D_CHOA CHU KANG AVE 5_4 ROOM_,487C_CHOA CHU KANG AVE 5_4 ROOM_,1,5,4,2,3,1,2,3,3,5,-0.007432,0.045189,0.002782,0.003259,0.018541,0.18,-0.029461,0.000000,0.000000,31.372791,-2.743423,-2,-90.378826,-48.924209,69.576533,0,-64.623758,0,-13.035337,111.035562,-11.754125,-75.074302,9.0,-2.916667,-1.0,116.399327,83.598932
8,U1,688B_CHOA CHU KANG DR_4 ROOM_,408_CHOA CHU KANG AVE 3_4 ROOM_,0,5,4,2,3,1,2,3,3,5,-0.053092,-0.245628,-0.125611,0.012094,-0.489993,-0.06,0.089226,0.000000,0.000000,194.597926,29.613549,-15,491.255596,286.058874,178.341982,0,522.272731,-1,-48.375041,1511.543389,1516.506134,502.587157,-3.0,8.833333,-14.0,1175.271761,-701.527346
9,U1,226A_COMPASSVALE WALK_4 ROOM_,122_SERANGOON NTH AVE 1_4 ROOM_,1,5,4,2,3,1,2,3,3,5,0.315450,0.006200,0.248355,-0.365496,-0.113842,0.18,0.150673,0.000000,-0.007007,-1235.511285,-43.811809,8,-12.399095,-379.131757,-417.612964,2,-382.405988,1,54.896769,1407.085519,352.764713,40.062867,9.0,14.916667,3.0,3182.606685,70.0683

In [77]:
df_pairwise.to_csv("df_pairwise.csv")